## Imports

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import numpy as np
import pandas as pd
import optuna

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    mean_absolute_percentage_error
)

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam

pd.set_option('display.max_columns', None)

In [ ]:
from optuna.visualization import (plot_optimization_history, plot_param_importances, plot_parallel_coordinate, plot_slice, plot_contour)

def exibir_graficos_optuna(study):
    
    # Gráfico de histórico da otimização - mostra a convergência do score ao longo dos trials
    fig = plot_optimization_history(study)
    fig.show()

    # Gráfico de importância dos parâmetros
    fig = plot_param_importances(study)
    fig.show()

    # Gráfico do efeito individual de cada parâmetro
    fig = plot_slice(study)
    fig.show()

## Base de dados

In [3]:
save_folder_path_refined = 'dados/refined'
df = pd.read_parquet(f"{save_folder_path_refined}/tb_analitica_ABEV3.parquet", engine = 'pyarrow')
df = df.sort_values("date").reset_index(drop=True)

In [4]:
df.columns

Index(['ticker', 'date', 'weekday', 'close', 'high', 'low', 'open', 'volume',
       'close_dolar', 'close_ibovespa', 'close_sp_500', 'selic', 'ipca',
       'ma20', 'ma50', 'bb_upper', 'bb_lower', 'rsi_wilder', 'macd',
       'macd_signal', 'weekday_sin', 'weekday_cos', 'month_sin', 'month_cos'],
      dtype='str')

In [16]:
df.shape[0]

2469

In [10]:
df.head(3)

,ticker,date,weekday,close,high,low,open,volume,close_dolar,close_ibovespa,close_sp_500,selic,ipca,ma20,ma50,bb_upper,bb_lower,rsi_wilder,macd,macd_signal,weekday_sin,weekday_cos,month_sin,month_cos
9,ABEV3,2016-07-01,Friday,12.809154,12.809154,12.655871,12.715851,22773000.0,3.2059,52233.0,2102.949951,0.052531,0.52,NaN,NaN,NaN,NaN,NaN,0.000000,0.000000,-0.951057,0.309017,1.224647e-16,-1.0
10,ABEV3,2016-07-04,Monday,12.702519,12.855804,12.649204,12.809151,6702100.0,3.2306,52569.0,2102.949951,0.052531,0.52,NaN,NaN,NaN,NaN,0.000000,-0.008506,-0.001701,0.000000,1.000000,1.224647e-16,-1.0
11,ABEV3,2016-07-05,Tuesday,12.855803,12.875796,12.635874,12.642539,13405900.0,3.2885,51842.0,2088.550049,0.052531,0.52,NaN,NaN,NaN,NaN,9.956564,-0.002846,-0.001930,0.951057,0.309017,1.224647e-16,-1.0


## Preparo dos dados

In [5]:
# ==============================================================================
# Features utilizadas
# ==============================================================================

features = [
    'close', 'high', 'low', 'open', 'volume',
    'close_dolar', 'close_ibovespa', 'close_sp_500', 'selic', 'ipca',
    'ma20', 'ma50', 'bb_upper', 'bb_lower', 'rsi_wilder', 'macd',
    'macd_signal', 'weekday_sin', 'weekday_cos', 'month_sin', 'month_cos'
]

# ==============================================================================
# Selecionar apenas as features
# ==============================================================================

data = df[features].copy()

# ==============================================================================
# Normalização
# ==============================================================================

scaler = MinMaxScaler()
scaled_data = scaler.fit_transform(data)

## Funções para orquestração do modelo

In [8]:
# ==============================================================================
# Orquestração do modelo
# ==============================================================================

def build_lstm_sequences(window_size, scaled_data):

    X = []
    y = []

    for i in range(window_size, len(scaled_data)):

        # últimos WINDOW_SIZE dias
        X.append(scaled_data[i-window_size:i])

        # target = close do dia atual
        y.append(scaled_data[i, 0])

    X = np.array(X)
    y = np.array(y)

    print(X.shape)
    print(y.shape)

    return X, y

def split_temporal(X, y, train_size = 0.70, valid_size = 0.15):

    n = len(X)

    train_end = int(n * train_size)
    valid_end = int(n * (train_size + valid_size))

    X_train = X[:train_end]
    y_train = y[:train_end]

    X_valid = X[train_end:valid_end]
    y_valid = y[train_end:valid_end]

    X_test = X[valid_end:]
    y_test = y[valid_end:]

    print(f"Treino     : {X_train.shape}")
    print(f"Validação  : {X_valid.shape}")
    print(f"Teste      : {X_test.shape}")

    return X_train, y_train, X_valid, y_valid, X_test, y_test

def generate_lstm_model(units_1, dropout, units_2, learning_rate, X_train):
    
    model = Sequential()

    model.add(LSTM(units=units_1,
                   return_sequences=True, 
                   input_shape=(X_train.shape[1], X_train.shape[2])))
    model.add(Dropout(dropout))
    model.add(LSTM(units=units_2))
    model.add(Dropout(0.2))
    model.add(Dense(16, activation="relu"))
    model.add(Dense(1))

    model.compile(optimizer=Adam(learning_rate = learning_rate),
                loss="mse", 
                metrics=["mae"])

    return model

## Optuna - Definição de hiperparâmetros

In [ ]:
# ==============================================================================
# Optuna - Definição dos hiperparâmetros
# ==============================================================================

for w in [30, 60, 90, 120]:

    window_size = w

    X, y = build_lstm_sequences(window_size, scaled_data)
    X_train, y_train, X_valid, y_valid, X_test, y_test = split_temporal(X, y)

    def objective(trial):

        # Hiperparâmetros
        units_1 = trial.suggest_int("units_1", 32, 128, step=32)
        units_2 = trial.suggest_int("units_2", 16, 64, step=16)
        dropout = trial.suggest_float("dropout", 0.10, 0.50)
        learning_rate = trial.suggest_float("learning_rate", 1e-4, 1e-2, log=True)
        batch_size = trial.suggest_categorical("batch_size", [16, 32, 64])

        # Modelo
        model = generate_lstm_model(units_1, dropout, units_2, learning_rate, X_train)

        history = model.fit(
            X_train, 
            y_train,
            validation_data=(X_valid, y_valid),
            epochs=100,
            batch_size=batch_size,
            callbacks=[EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True)],
            verbose=0
        )

        return min(history.history["val_loss"])
    
    study = optuna.create_study(direction="minimize", study_name=f"LSTM Stock Prediction | window_size = {w}")
    study.optimize(objective, n_trials=30, show_progress_bar=True)

    print("=" * 60)
    print(f"Melhores hiperparâmetros | window_size = {w}")
    print("=" * 60)

    for k, v in study.best_params.items():
        print(f"{k}: {v}")

    print("\nMelhor validation loss:")
    print(study.best_value)

[I 2026-07-05 20:54:02,045] A new study created in memory with name: LSTM Stock Prediction | window_size = 30


(2439, 30, 21)
(2439,)
Treino     : (1707, 30, 21)
Validação  : (366, 30, 21)
Teste      : (366, 30, 21)


  0%|          | 0/30 [00:00<?, ?it/s]

g:\Meu Drive\5. Cursos\Pós ML Engineering\Fase 4 - Deep Learning e IA\lstm-stock-predictor-api\.venv\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


[I 2026-07-05 20:54:26,432] Trial 0 finished with value: 0.0062607950530946255 and parameters: {'units_1': 128, 'units_2': 32, 'dropout': 0.13926176346345023, 'learning_rate': 0.0007582621084246966, 'batch_size': 32}. Best is trial 0 with value: 0.0062607950530946255.
[I 2026-07-05 20:55:58,449] Trial 1 finished with value: 0.014625588431954384 and parameters: {'units_1': 64, 'units_2': 32, 'dropout': 0.439260451783687, 'learning_rate': 0.00010879569303872893, 'batch_size': 64}. Best is trial 0 with value: 0.0062607950530946255.
[I 2026-07-05 20:56:14,745] Trial 2 finished with value: 0.007063230499625206 and parameters: {'units_1': 96, 'units_2': 64, 'dropout': 0.4106357263990432, 'learning_rate': 0.005845225236760581, 'batch_size': 64}. Best is trial 0 with value: 0.0062607950530946255.
[I 2026-07-05 20:56:48,817] Trial 3 finished with value: 0.007106476929038763 and parameters: {'units_1': 128, 'units_2': 64, 'dropout': 0.399160196469696, 'learning_rate': 0.004287216514325487, 'batc

In [ ]:
exibir_graficos_optuna(study)

## Modelo com melhores hiperparâmetros

In [ ]:
# ==============================================================================
# Definição do modelo
# ==============================================================================

# best = study.best_params

units_1 = best["units_1"]
dropout = best["dropout"]
units_2 = best["units_2"]
learning_rate = best["learning_rate"]
batch_size = best["batch_size"]
# window_size = 60

X, y = build_lstm_sequences(window_size, scaled_data)
X_train, y_train, X_valid, y_valid, X_test, y_test = split_temporal(X, y)

model = generate_lstm_model(units_1, dropout, units_2, learning_rate, X_train)

print(model.summary())

history = model.fit(
    X_train, 
    y_train,
    validation_data=(X_valid, y_valid),
    epochs=100,
    batch_size=batch_size,
    callbacks=[EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True)],
    verbose=1
)

In [ ]:
# ==============================================================================
# Predição
# ==============================================================================

pred_scaled = model.predict(X_test)

# ==============================================================================
# Desnormalização
# ==============================================================================

dummy_pred = np.zeros((len(pred_scaled), len(features)))
dummy_real = np.zeros((len(y_test), len(features)))

dummy_pred[:,0] = pred_scaled.flatten()
dummy_real[:,0] = y_test

pred = scaler.inverse_transform(dummy_pred)[:,0]
real = scaler.inverse_transform(dummy_real)[:,0]

# ==============================================================================
# Métricas
# ==============================================================================

mae = mean_absolute_error(real, pred)
rmse = np.sqrt(mean_squared_error(real, pred))

mape = mean_absolute_percentage_error(real, pred)

print(f"MAE : {mae:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"MAPE: {mape:.2%}")